# BulkFormer Embedding Extraction 

In this notebook we extract RNA-seq embeddings for our TCGA dataset using the pretrained **BulkFormer-147M** model.
The model was pretrained on ~500,000 bulk RNA-seq samples from public repositories and learns rich
representations of gene expression patterns. By running our TCGA samples through the frozen model we obtain
a **643-dimensional embedding vector per sample** that captures the biologically meaningful structure of
the expression profile. These embeddings will later be fused with DNA methylation embeddings for
multi-omics cancer-type classification.

**Key concepts used in this notebook:**
- **Ensembl gene ID** — standardized gene identifier (e.g. `ENSG00000141510` for TP53). BulkFormer
  operates on exactly 20,010 of these IDs as its fixed vocabulary.
- **TPM (Transcripts Per Million)** — a normalization that corrects for gene length and sequencing depth,
  making samples directly comparable across experiments.
- **log-TPM** — we apply `log(TPM + 1)` to compress the wide dynamic range of expression values.
  The `+1` avoids `log(0)` for silent genes.
- **Embedding** — a fixed-length float vector produced by the neural network. Similar samples produce
  similar vectors; the model has learned what "similar" means from 500k training samples.
- **Frozen / eval mode** — we use the model purely as a feature extractor. No weights are updated:
  `model.eval()` disables dropout and `torch.no_grad()` skips gradient computation.


## Install torch-geometric

BulkFormer uses `torch-geometric`'s `SparseTensor` to represent the gene-gene interaction graph
efficiently. Most gene pairs don't interact, so storing only the non-zero edges (a sparse format)
saves memory compared to a full 20,010 × 20,010 dense matrix. This package is not in the project's
`requirements.txt`, so we install it once here.

The cell auto-detects the PyTorch + CUDA versions to fetch the matching pre-built wheels.

In [2]:
import subprocess, sys, torch

cuda_ver  = torch.version.cuda
torch_ver = torch.__version__.split('+')[0]
print(f'PyTorch {torch_ver}, CUDA {cuda_ver}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)

pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver.replace(".","")}.html'
for pkg in ['torch-scatter', 'torch-sparse']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg, '-f', pyg_url], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'performer-pytorch'], check=True)

print('Installation complete — RESTART THE KERNEL before continuing.')

PyTorch 2.6.0, CUDA 12.4
Installation complete — RESTART THE KERNEL before continuing.


## Set the working directory, paths, and GPU

BulkFormer's internal imports use relative paths (`from utils.BulkFormer import BulkFormer`,
`from model.config import model_params`), so Python's working directory must be the root of the
BulkFormer folder while those imports are resolved.

We also set `PROJECT_ROOT` to point back to the project folder so we can reference the TCGA data
and write outputs without hard-coding long absolute paths everywhere. GPU 0 is selected via
`CUDA_VISIBLE_DEVICES`.

In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT    = Path(r'c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer')
PROJECT_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f'Working directory : {os.getcwd()}')
print(f'Project root      : {PROJECT_ROOT}')
print(f'Python            : {sys.version}')
assert REPO_ROOT.exists(), f'BulkFormer folder not found at {REPO_ROOT} — run scripts/setup_bulkformer.py first.'

Working directory : c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer
Project root      : c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification
Python            : 3.11.7 (tags/v3.11.7:fa7a6f2, Dec  4 2023, 19:24:49) [MSC v.1937 64 bit (AMD64)]


## Import all required libraries

In [2]:
import math
import anndata
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch_sparse import SparseTensor

from utils.BulkFormer import BulkFormer
from model.config import model_params

print('All imports successful.')

c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\venv_bulkformer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


## Configure data paths and device

We define all file paths relative to `REPO_ROOT` (for files inside the BulkFormer folder) and
`PROJECT_ROOT` (for the TCGA data and output files). This keeps things portable and easy to audit.

The three graph/embedding files the model needs at construction time:
- **`G_tcga.pt`** — a sparse adjacency matrix (source/target gene index pairs) encoding known
  gene-gene co-expression relationships from TCGA. The model's graph neural network layer uses
  this to let each gene attend to its neighbours' expression values.
- **`G_tcga_weight.pt`** — a scalar weight for each edge (how strongly two genes co-vary).
- **`esm2_feature_concat.pt`** — protein-sequence embeddings from Meta's ESM2 model, one vector
  per gene. This gives the model information about the protein each gene encodes, complementing
  the expression-level signal.

Running on `cuda` is required for BulkFormer because some of its CUDA-specific operations are
not implemented for CPU.

In [3]:
device = 'cuda'

graph_path    = REPO_ROOT / 'data' / 'G_tcga.pt'
weights_path  = REPO_ROOT / 'data' / 'G_tcga_weight.pt'
gene_emb_path = REPO_ROOT / 'data' / 'esm2_feature_concat.pt'
int_gene_path = REPO_ROOT / 'data' / 'interested_gene_list.pt'
gene_info_csv = REPO_ROOT / 'data' / 'bulkformer_gene_info.csv'
gene_len_csv  = REPO_ROOT / 'data' / 'gene_length_df.csv'
ckpt_path     = PROJECT_ROOT / 'pretrained_models' / 'BulkFormer_147M.pt'
tcga_h5ad     = PROJECT_ROOT / 'data' / 'processed' / 'tcga_rna_seq.h5ad'
output_path   = PROJECT_ROOT / 'data' / 'processed' / 'bulkformer_embeddings.npy'

required = {
    'Gene graph'        : graph_path,
    'Graph weights'     : weights_path,
    'ESM2 embeddings'   : gene_emb_path,
    'Interested genes'  : int_gene_path,
    'Gene info CSV'     : gene_info_csv,
    'Gene length CSV'   : gene_len_csv,
    'Model checkpoint'  : ckpt_path,
    'TCGA RNA-seq h5ad' : tcga_h5ad,
}
all_ok = True
for name, path in required.items():
    status = '✓' if path.exists() else '✗  MISSING'
    print(f'  {status}  {name}: {path}')
    if not path.exists():
        all_ok = False
print('\nAll files found — ready to proceed.' if all_ok else '\nSome files are missing — see Prerequisites.')

  ✓  Gene graph: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\G_tcga.pt
  ✓  Graph weights: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\G_tcga_weight.pt
  ✓  ESM2 embeddings: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\esm2_feature_concat.pt
  ✓  Interested genes: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\interested_gene_list.pt
  ✓  Gene info CSV: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\bulkformer_gene_info.csv
  ✓  Gene length CSV: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\BulkFormer\data\gene_length_df.csv
  ✓  Model checkpoint: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\pretrained_models\BulkFormer_147M.pt
  ✓  TCGA RNA-seq h5ad: c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\data\processed\tcga_rna_seq.h5ad

All files found 

## Initialize the BulkFormer model

Here we build the model skeleton before loading any weights.

The initialization requires two external data structures that are baked into the architecture:
1. **Gene-gene interaction graph** — loaded as two index tensors (source and target gene indices)
   plus a weight tensor, then combined into a `SparseTensor`. The `.t()` call transposes it
   to match the convention expected by the graph attention layer. The whole thing goes to GPU.
2. **ESM2 protein embeddings** — one vector per gene, capturing protein-sequence information.
   The model uses these as positional encodings so each gene starts with a biologically informed
   representation rather than a random one.

The `model_params` dict from `config.py` specifies the 147M architecture:
- `dim = 640` — hidden dimension; every intermediate gene representation is 640 numbers wide.
- `gene_length = 20010` — fixed vocabulary size.
- `p_repeat = 12` — 12 Performer (linearised attention) encoder layers.
- `gb_repeat = 1` — 1 graph-neural-network + Performer block.

We print the total parameter count as a sanity check.

In [4]:
graph   = torch.load(graph_path,   map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph   = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)

gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)

model_params['graph']    = graph
model_params['gene_emb'] = gene_emb
model = BulkFormer(**model_params).to(device)

print(f'Model initialized.  Parameters: {sum(p.numel() for p in model.parameters()):,}')

Model initialized.  Parameters: 132,143,047


## Load pretrained weights (frozen, eval mode)

We load the actual learned weights from the checkpoint file.

Two things to handle:
1. **`module.` prefix stripping** — the checkpoint was saved from a `DataParallel`-wrapped
   model (multi-GPU training), which prepends `module.` to every key. We strip it so the
   keys match the single-GPU model constructed above.
2. **Freezing** — after loading, we call `model.eval()` to switch off dropout and
   set `requires_grad = False` on every parameter. This ensures no weights change during
   inference — we use BulkFormer purely as a fixed feature extractor.

In [5]:
ckpt_model = torch.load(ckpt_path, weights_only=False, map_location='cpu')

new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith('module.') else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict)
model.eval()
for param in model.parameters():
    param.requires_grad = False

print('Pretrained weights loaded.  Model is FROZEN (eval mode, no gradients).')

Pretrained weights loaded.  Model is FROZEN (eval mode, no gradients).


## TPM normalization function

BulkFormer expects log-TPM values as input, not raw read counts. Raw counts are not
directly comparable across samples because:
- **Gene length bias** — longer genes capture more sequencing reads at the same expression level.
- **Sequencing depth bias** — samples sequenced more deeply have larger total counts.

TPM (Transcripts Per Million) corrects both:
1. Divide each gene's count by its length in kilobases → rate.
2. Divide by the sum of all rates per sample, multiply by 1,000,000 → TPM.
3. Apply `log(TPM + 1)` → log-TPM (compresses the right-skewed distribution; `+1` avoids log(0)).

Gene lengths (in base pairs) come from `gene_length_df.csv` shipped with the BulkFormer repo.
Genes absent from that file fall back to 1,000 bp as a neutral default.

In [6]:
def normalize_data(X_df, gene_length_dict):
    gene_names      = X_df.columns
    gene_lengths_kb = np.array([gene_length_dict.get(g, 1000) / 1000 for g in gene_names])
    counts          = X_df.values

    rate            = counts / gene_lengths_kb
    sums            = rate.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1e-6
    tpm             = rate / sums * 1e6
    log_tpm         = np.log1p(tpm)

    return pd.DataFrame(log_tpm, index=X_df.index, columns=X_df.columns)

print('normalize_data() defined.')

normalize_data() defined.


## Gene-alignment function

BulkFormer always processes exactly 20,010 genes in a fixed order — its learned vocabulary.
The TCGA data contains a different (and overlapping) set of Ensembl IDs, so alignment is required.

This function:
1. Identifies vocabulary genes absent from our data.
2. Fills those positions with **−10** — a sentinel value that tells the model "this gene
   was not measured". The model has a masking mechanism that learns to treat −10 entries
   differently from real expression values.
3. Reorders columns to match the vocabulary order exactly.
4. Builds a binary mask (`1 = imputed`, `0 = real`) so we can compute the **gene-missing rate**
   (`mask_prob`) — the fraction of vocabulary genes absent from our dataset. BulkFormer uses
   this scalar as an auxiliary input feature appended to each sample's embedding.

In [7]:
def main_gene_selection(X_df, gene_list):
    to_fill = list(set(gene_list) - set(X_df.columns))
    padding = pd.DataFrame(
        np.full((X_df.shape[0], len(to_fill)), -10),
        columns=to_fill, index=X_df.index
    )
    X_df = pd.concat([X_df, padding], axis=1)[gene_list]
    var  = pd.DataFrame({'mask': [1 if g in to_fill else 0 for g in gene_list]}, index=gene_list)
    return X_df, to_fill, var

print('main_gene_selection() defined.')

main_gene_selection() defined.


## Feature extraction function

This is the core inference loop. For each batch of samples it:
1. Converts the expression array to a PyTorch tensor and moves it to GPU.
2. Calls `model(X)` — the forward pass produces per-gene embeddings of shape
   `[batch, 20010, 643]` (640 hidden dimensions + 3 auxiliary scalars appended by the model:
   mask probability, mean expression, and non-zero gene ratio).
3. Optionally restricts to a subset of "interested genes" before aggregation.
4. Aggregates across genes to produce one vector per sample.

We use **`aggregate_type='mean'`** (average over all interested gene embeddings), which is the
approach used in the official BulkFormer publication.

We also use `torch.amp.autocast` (automatic mixed precision) for a free speed-up on the RTX 4050:
it computes attention in float16 where safe, then accumulates in float32, roughly doubling
throughput with no accuracy loss.

In [8]:
def extract_feature(expr_array, aggregate_type, device, batch_size,
                    mask_prob=0.1, interested_gene_idx=None):
    loader  = DataLoader(TensorDataset(torch.tensor(expr_array, dtype=torch.float32)),
                         batch_size=batch_size, shuffle=False)
    model.eval()
    all_emb = []

    with torch.no_grad(), torch.amp.autocast('cuda', enabled=True):
        for (X,) in tqdm(loader, desc='Extracting embeddings'):
            X   = X.to(device)
            emb = model(X, mask_prob=mask_prob, output_expr=False).detach().cpu().numpy()

            if interested_gene_idx is not None:
                emb = emb[:, interested_gene_idx, :]

            if aggregate_type == 'max':
                final = np.max(emb, axis=1)
            elif aggregate_type == 'median':
                final = np.median(emb, axis=1)
            elif aggregate_type == 'all':
                final = np.max(emb, axis=1) + np.mean(emb, axis=1) + np.median(emb, axis=1)
            else:
                final = np.mean(emb, axis=1)

            all_emb.append(final)
            del emb
            torch.cuda.empty_cache()

    return torch.tensor(np.vstack(all_emb), dtype=torch.float32)

print('extract_feature() defined.')

extract_feature() defined.


## Load TCGA RNA-seq data

We load the preprocessed TCGA dataset from `data/processed/tcga_rna_seq.h5ad`.
This file was created by `preprocess_colab.ipynb`, which assembled raw STAR gene-count TSV
files (one per case) into a single **AnnData** object.

**AnnData** is the standard container for genomics matrices in Python:
- `.X` — the expression matrix (samples × genes), stored as raw read counts.
- `.obs` — sample-level metadata: `case_id`, `cancer_type`, `file_id`.
- `.var` — gene-level metadata: `gene_name`, `gene_type`; indexed by Ensembl ID.

We convert `.X` to a dense pandas DataFrame (samples × Ensembl IDs) to pass into the
normalization pipeline.

In [9]:
adata = anndata.read_h5ad(tcga_h5ad)
print(f'Loaded TCGA RNA-seq data')
print(f'  Samples  : {adata.shape[0]}')
print(f'  Genes    : {adata.shape[1]}')
print(f'  Cancer types: {adata.obs["cancer_type"].value_counts().to_dict()}')

X_raw = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
count_df = pd.DataFrame(X_raw, index=adata.obs_names, columns=adata.var_names)
print(f'\nRaw count matrix shape: {count_df.shape}')
print(f'Sample (first 3 rows, first 4 genes):')
print(count_df.iloc[:3, :4])

Loaded TCGA RNA-seq data
  Samples  : 800
  Genes    : 60616
  Cancer types: {'TCGA-BRCA': 134, 'TCGA-LUAD': 134, 'TCGA-COAD': 133, 'TCGA-KIRC': 133, 'TCGA-LIHC': 133, 'TCGA-THCA': 133}

Raw count matrix shape: (800, 60616)
Sample (first 3 rows, first 4 genes):
   ENSG00000000003  ENSG00000000005  ENSG00000000419  ENSG00000000457
0              850                5             1680             1559
1             4148              604             1194              996
2             2842              113             1617             1358


## Normalize raw counts to log-TPM

We apply the `normalize_data()` function defined above to convert raw read counts to
log(TPM + 1) values. Gene lengths (in base pairs) are loaded from `gene_length_df.csv`,
which is included in the BulkFormer repository.

After normalization, all values should be non-negative, and the range will be roughly
[0, ~15] depending on the highest-expressed genes.

In [10]:
gene_len_df   = pd.read_csv(gene_len_csv)
gene_len_dict = gene_len_df.set_index('ensg_id')['length'].to_dict()
print(f'Gene lengths loaded: {len(gene_len_dict):,} genes')

log_tpm_df = normalize_data(X_df=count_df, gene_length_dict=gene_len_dict)
print(f'Log-TPM matrix shape : {log_tpm_df.shape}')
print(f'Value range          : [{log_tpm_df.values.min():.2f}, {log_tpm_df.values.max():.2f}]')

Gene lengths loaded: 78,932 genes
Log-TPM matrix shape : (800, 60616)
Value range          : [0.00, 12.97]


## Align expression data to BulkFormer's gene vocabulary

The TCGA dataset uses Ensembl IDs (without version suffixes, e.g. `ENSG00000141510`) but
covers a different set of genes than BulkFormer's fixed 20,010-gene vocabulary.
We use `main_gene_selection()` to:
- Pad missing genes with the −10 sentinel.
- Reorder columns to match the exact vocabulary order.
- Compute `mask_prob` — the fraction of vocabulary genes we don't have data for.
  BulkFormer appends this value as one of the three auxiliary scalars in the output embedding,
  so it must be computed per-dataset and passed to `extract_feature()`.

In [11]:
bulkformer_info  = pd.read_csv(gene_info_csv)
bulkformer_genes = bulkformer_info['ensg_id'].tolist()
print(f'BulkFormer vocabulary size: {len(bulkformer_genes):,} genes')

input_df, filled_genes, var = main_gene_selection(log_tpm_df, bulkformer_genes)
mask_prob = var['mask'].mean()

print(f'Aligned matrix shape : {input_df.shape}')
print(f'Genes filled (−10)   : {len(filled_genes):,}')
print(f'Gene-missing rate    : {mask_prob:.4f}  ({mask_prob*100:.1f}% of vocabulary)')

BulkFormer vocabulary size: 20,010 genes
Aligned matrix shape : (800, 20010)
Genes filled (−10)   : 103
Gene-missing rate    : 0.0051  (0.5% of vocabulary)


## Load "interested gene" indices

The `interested_gene_list.pt` file contains a list of integer indices into the 20,010-gene
vocabulary. Rather than averaging all 20,010 per-gene embeddings into one sample vector,
the BulkFormer authors aggregate only over this curated subset — typically protein-coding
genes with reliable measurements and biological relevance. We follow the same approach to
stay consistent with the published methodology.

In [12]:
interested_gene_idx = torch.load(int_gene_path, weights_only=False)
print(f'Interested gene indices: {len(interested_gene_idx):,}')
print(f'First 10 indices: {interested_gene_idx[:10]}')

Interested gene indices: 2,000
First 10 indices: [16385, 10768, 5261, 391, 16614, 1862, 15910, 3850, 18456, 10758]


## Extract embeddings for all TCGA samples

We now run the full extraction loop over all samples in the TCGA dataset.

**Batch size** — we use `batch_size=8` which fits comfortably in the 6 GB VRAM of the
RTX 4050. Each forward pass processes 8 samples × 20,010 genes × 643 hidden dimensions
simultaneously. Larger batch sizes would be faster but risk OOM errors; smaller batches
are safer but slower. If CUDA out-of-memory errors occur, reduce to 4.

**Expected runtime** — at batch_size=8 on an RTX 4050 (~70 TFLOPS FP16 with AMP):
~800 TCGA samples should complete in roughly **8–15 minutes**.

The output tensor has shape `(N_samples, 643)` — one 643-dim float32 vector per sample.

In [13]:
BATCH_SIZE = 8   # reduce to 4 if you get CUDA out-of-memory

print(f'Extracting BulkFormer embeddings for {len(input_df)} samples...')
print(f'Batch size: {BATCH_SIZE}  |  Device: {device}')

embeddings = extract_feature(
    expr_array          = input_df.values,
    aggregate_type      = 'mean',
    device              = device,
    batch_size          = BATCH_SIZE,
    mask_prob           = mask_prob,
    interested_gene_idx = interested_gene_idx,
)

print(f'\nEmbedding matrix shape : {embeddings.shape}')   # (N_samples, 643)
print(f'Dtype                  : {embeddings.dtype}')
print(f'Value range            : [{embeddings.min():.4f}, {embeddings.max():.4f}]')
print(f'Mean / Std             : {embeddings.mean():.4f} / {embeddings.std():.4f}')

Extracting BulkFormer embeddings for 800 samples...
Batch size: 8  |  Device: cuda


Extracting embeddings: 100%|██████████| 100/100 [45:59<00:00, 27.59s/it]


Embedding matrix shape : torch.Size([800, 643])
Dtype                  : torch.float32
Value range            : [-2.3323, 2.4708]
Mean / Std             : 0.0045 / 0.5230


## Validate and save embeddings

- Shape must be `(N_samples, 643)` — the 640 hidden dims + 3 auxiliary scalars.
- No NaN or Inf values — these would propagate silently into the downstream classifier.
- dtype must be float32 for compatibility with scikit-learn and other classifiers.

We also save a `bulkformer_meta.csv` alongside
so we can reconstruct sample IDs and labels when loading later.

In [14]:
N = embeddings.shape[0]

assert embeddings.shape == (N, 643), f'Unexpected shape: {embeddings.shape}'
assert not torch.isnan(embeddings).any(), 'Embedding contains NaN values!'
assert not torch.isinf(embeddings).any(), 'Embedding contains Inf values!'
assert embeddings.dtype == torch.float32
print('Sanity checks passed.')

emb_np = embeddings.numpy()
np.save(output_path, emb_np)
print(f'Saved embeddings to : {output_path}')
print(f'Shape               : {emb_np.shape}')

meta_path = output_path.parent / 'bulkformer_meta.csv'
meta_df   = adata.obs[['case_id', 'cancer_type']].copy().reset_index(drop=True)
meta_df.to_csv(meta_path, index=False)
print(f'Saved metadata to   : {meta_path}')
print(f'\nCancer type counts:')
print(meta_df['cancer_type'].value_counts())

Sanity checks passed.
Saved embeddings to : c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\data\processed\bulkformer_embeddings.npy
Shape               : (800, 643)
Saved metadata to   : c:\Users\Bojana\Desktop\Proekti\multi-omics-cancer-classification\data\processed\bulkformer_meta.csv

Cancer type counts:
cancer_type
TCGA-BRCA    134
TCGA-LUAD    134
TCGA-COAD    133
TCGA-KIRC    133
TCGA-LIHC    133
TCGA-THCA    133
Name: count, dtype: int64


## Summary

In this notebook we completed the BulkFormer embedding extraction pipeline:

1. Loaded the pretrained **BulkFormer-147M** model in frozen eval mode.
2. Converted our TCGA raw read counts to **log-TPM** (the model's expected input format).
3. Aligned our gene set to BulkFormer's fixed **20,010-gene vocabulary**, filling missing genes
   with the −10 sentinel and computing the gene-missing rate.
4. Ran inference in batches, aggregating per-gene embeddings via **mean pooling** over the
   interested gene subset.
5. Saved the resulting **(N_samples, 643)** embedding matrix to
   `data/processed/bulkformer_embeddings.npy`.

**Key numbers:**
- BulkFormer vocabulary: **20,010** Ensembl gene IDs
- Input per sample: log-TPM vector of length 20,010
- Output per sample: **643-dimensional** float32 vector
  (640 hidden dims + 3 auxiliary scalars: mask rate, mean expression, non-zero gene ratio)